In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
project_path=os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)
from utils.transformations import reusable 



In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "200")

In [0]:

df_check = spark.read \
    .format("parquet") \
    .load("abfss://bronze@storageazuresprojectzain.dfs.core.windows.net/DimUser")

print("Total rows:", df_check.count())
df_check.show(truncate=False)

## **DimUser**


## **AUTOLOADER**

In [0]:
df_user = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimUser/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storageazuresprojectzain.dfs.core.windows.net/DimUser")
)


In [0]:
df_user=df_user.withColumn("user_name",upper(col("user_name")))
df_user_obj=reusable()
df_user=df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user=df_user.dropDuplicates(["user_id"])



In [0]:
df_user.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimUser/data")\
    .toTable('`spotify-cata`.`silver`.`Dimuser`')


## **DimArtist**

In [0]:
df_artist = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimArt/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storageazuresprojectzain.dfs.core.windows.net/DimArtist")
)


In [0]:
df_artist_obj=reusable()
df_artist=df_artist_obj.dropColumns(df_artist,['_rescued_data'])
df_artist=df_artist.dropDuplicates(["artist_id"])



In [0]:
df_artist.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimArt/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimArt/data")\
    .toTable('`spotify-cata`.`silver`.`DimArtist`')
    

## **DimTrack**

In [0]:
df_track = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/Dimtrack/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storageazuresprojectzain.dfs.core.windows.net/DimTrack")
)


In [0]:
df_track=df_track.withColumn("durationFlag",when(col('duration_sec')<150,"LOW")\
                                      .when(col('duration_sec')<300,"MEDIUM")\
                                      .otherwise("HIGH"))
df_track=df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))
df_track=reusable().dropColumns(df_track,['_rescued_data'])


In [0]:
df_track.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimTrack/data")\
    .toTable('`spotify-cata`.`silver`.`DimTrack`')
    

## **DimDate**

In [0]:
df_date = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimDate/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storageazuresprojectzain.dfs.core.windows.net/DimDate")
)


In [0]:
df_date=reusable().dropColumns(df_date,['_rescued_data'])


In [0]:
df_date.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazuresprojectzain.dfs.core.windows.net/DimDate/data")\
    .toTable('`spotify-cata`.`silver`.`DimDate`')
    

## **FactStream**

In [0]:
df_fact = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/FactStream/checkpoint")\
        .option("schemaEvolutionMode","addNewColumns")\
        .load("abfss://bronze@storageazuresprojectzain.dfs.core.windows.net/FactStream")
)


In [0]:
df_fact=reusable().dropColumns(df_fact,['_rescued_data'])

df_fact.writeStream\
    .format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@storageazuresprojectzain.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path","abfss://silver@storageazuresprojectzain.dfs.core.windows.net/FactStream/data")\
    .toTable('`spotify-cata`.`silver`.`FactStream`')

